# Observing effects of L2 penalty in polynomial regression

In this assignment, we will run ridge regression multiple times with different L2 penalties to see which one produces the best fit. We will revisit the example of polynomial regression as a means to see the effect of L2 regularization. In particular, we will:
- Use a pre-built implementation of regression to run polynomial regression
- Use matplotlib to visualize polynomial regressions
- Use a pre-built implementation of regression to run polynomial regression, this time with L2 penalty
- Use matplotlib to visualize polynomial regressions under L2 regularization
- Choose best L2 penalty using cross-validation.
- Assess the final fit using test data.

We will continue to use the House data from previous assignments.

In the next programming assignment for this module, we will implement our own ridge regression learning algorithm using gradient descent.

## Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge

## Helping Functions

Copy and paste an equivalent of ‘polynomial_sframe’ function from Module 3 (Polynomial Regression). This function accepts an array ‘feature’ (of type  pandas.Series) and a maximal ‘degree’ and returns an data frame (of type  pandas.DataFrame) with the first column equal to ‘feature’ and the remaining columns equal to ‘feature’ to increasing integer powers up to ‘degree’.

In [2]:
def add_powers(feature, power):
    ## i'll return a series instead of a dataframe
    return feature.astype(float) ** power

def polynomial_df(feature, degree):
    poly_df = pd.DataFrame()
    for i in range(1, degree + 1):
        poly_df[f'{feature.name}_{i}'] = add_powers(feature, i)
    return poly_df

## Loading Data

For the remainder of the assignment we will be working with the house Sales data as in Module 3 (Polynomial Regression). Load in the data and also sort the sales data frame by ‘sqft_living’. When we plot the fitted values we want to join them up in a line and this works best if the variable on the X-axis (which will be ‘sqft_living’) is sorted. For houses with identical square footage, we break the tie by their prices.

In [4]:
data = pd.read_csv("../../data/kc_house_data.csv")
train_data = pd.read_csv("../../data/wk3_kc_house_train_data.csv")
valid_data = pd.read_csv("../../data/wk3_kc_house_valid_data.csv")
test_data = pd.read_csv("../../data/wk3_kc_house_test_data.csv")
train_valid_shuffled = pd.read_csv("../../data/wk3_kc_house_train_valid_shuffled.csv")
set_1 = pd.read_csv("../../data/wk3_kc_house_set_1_data.csv")
set_2 = pd.read_csv("../../data/wk3_kc_house_set_2_data.csv")
set_3 = pd.read_csv("../../data/wk3_kc_house_set_3_data.csv")
set_4 = pd.read_csv("../../data/wk3_kc_house_set_4_data.csv")

for df in (data, train_data, test_data, valid_data, train_valid_shuffled, set_1, set_2, set_3, set_4):
    # Pandas operations return a new copy of the dataframe and leave the original one untouched
    # that's why you must add inplace=true
    df.sort_values(by=['sqft_living','price'], inplace=True)

## Polynomial Regression With Small L2 Penalty

Let us revisit the 15th-order polynomial model using the 'sqft_living' input. Generate polynomial features up to degree 15 using `polynomial_df()` and fit a model with these features. When fitting the model, use an L2 penalty of 1.5e-5

Note: When we have so many features and so few data points, the solution can become highly numerically unstable, which can sometimes lead to strange unpredictable results. Thus, rather than using no regularization, we will introduce a tiny amount of regularization (l2_penalty=1.5e-5) to make the solution numerically stable. (In lecture, we discussed the fact that regularization can also help with numerical stability, and here we are seeing a practical example.)

With the L2 penalty specified above, fit the model and print out the learned weights. Add "alpha=l2_small_penalty" and "normalize=True" to the parameter list of linear_model.Ridge

In [5]:
# defining our l2 penalty
l2_small_penalty = 1.5e-5

# getting the features list
poly15_data = polynomial_df(data['sqft_living'], 15)

# normalization: substract mean + divide by l2-norm (standard deviation)
# note: normalize=True is deprecated in newer versions of sklern
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
model = make_pipeline(
    StandardScaler(), # defaults to `with_mean=True, with_std=True`
    Ridge(alpha=l2_small_penalty) # defaults to fit_intercept=True
    )

# fitting the model
model.fit(poly15_data, data['price'])

,steps,"[('standardscaler', ...), ('ridge', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,alpha,1.5e-05
,fit_intercept,True
,copy_X,True
,max_iter,None


In [6]:
model.named_steps['ridge'].coef_

array([   125511.76199354,    514854.73305913,  -3856820.43047518,
        15258921.1192429 , -26619491.32677034,   7258939.31399683,
        24095277.21430915,  -1854300.13766676, -20494234.33833126,
       -10619604.25939411,   8719258.48160321,  14175686.59811249,
         2020108.37439909, -10873885.20844129,   2351299.70149168])

**Quiz Question**: What’s the learned value for the coefficient of feature power_1?

In [7]:
coefs = model.named_steps['ridge'].coef_
print ( np.min(coefs) )
print ( np.max(coefs) )
print ( "Power_1 coef:", coefs[1] )

-26619491.326770335
24095277.214309152
Power_1 coef: 514854.73305912805


## Observe Overfitting

Recall from Module 3 (Polynomial Regression) that the polynomial fit of degree 15 changed wildly whenever the data changed. In particular, when we split the sales data into four subsets and fit the model of degree 15, the result came out to be very different for each subset. The model had a high variance. We will see in a moment that ridge regression reduces such variance. But first, we must reproduce the experiment we did in Module 3.

Just as we did in Module 3 (Polynomial Regression), fit a 15th degree polynomial on each of the 4 sets, plot the results and view the weights for the four models. This time, set

In [9]:
l2_small_penalty=1e-9

Make sure to add "alpha=l2_small_penalty" and "normalize=True" to the parameter list of linear_model.Ridge.

The four curves should differ from one another a lot, as should the coefficients you learned.

In [10]:
# array to hold coefficients from all 4 models
all_coefs = []

# fitting on each subset
for df in (set_1, set_2, set_3, set_4):
    # getting the features list
    poly15_data = polynomial_df(df['sqft_living'], 15)
    y = df['price'] # watch not to define poly15_data['price'], we don't want to scale Target 

    # normalization: substract mean + divide by l2-norm (standard deviation)
    # note: normalize=True is deprecated in newer versions of sklern
    model = make_pipeline(
        StandardScaler(), # defaults to `with_mean=True, with_std=True`
        Ridge(alpha=l2_small_penalty) # defaults to fit_intercept=True
        )

    # fitting the model
    model.fit(poly15_data, y)

    # printing out the coefficients
    new_coefs = model.named_steps['ridge'].coef_
    all_coefs.append(new_coefs)
    print (new_coefs)

# stack all coefs
all_coefs = np.vstack(all_coefs)

[ 7.76452307e+05  1.37266687e+05 -4.38201266e+07  4.30948531e+08
 -2.12146884e+09  5.49217398e+09 -6.13178396e+09 -1.34581172e+09
  6.69066360e+09  2.50712412e+09 -6.09551288e+09 -6.09035394e+09
  3.38030622e+09  9.50790817e+09 -6.18159439e+09]
[-3.84821788e+06  3.70181638e+07 -1.69623784e+08  4.79554945e+08
 -7.60506968e+08  1.92948824e+08  1.44480842e+09 -1.71973007e+09
 -9.17980791e+08  1.91396312e+09  1.12822176e+09 -1.80583432e+09
 -1.43599726e+09  2.38945611e+09 -7.72171473e+08]
[ 2.70525056e+06 -2.06347406e+07  8.12413159e+07 -1.50483986e+08
 -2.11886476e+08  1.82328341e+09 -3.56713763e+09  1.52934891e+09
  3.08528622e+09 -2.01008488e+09 -3.17899242e+09  1.80560468e+09
  3.37446211e+09 -3.45390138e+09  8.91594149e+08]
[-6.69612369e+05  5.62808043e+06 -5.78068369e+06 -9.97780211e+07
  6.04038501e+08 -1.58428535e+09  1.87095298e+09 -8.22232304e+07
 -1.75440168e+09  3.93935915e+08  1.64989619e+09 -3.95660025e+08
 -1.62596232e+09  1.32790559e+09 -3.03379466e+08]


**Quiz Question**: For the models learned in each of these training sets, what are the smallest and largest values you learned for the coefficient of feature power_1? (For the purpose of answering this question, negative numbers are considered "smaller" than positive numbers. So -5 is smaller than -3, and -3 is smaller than 5 and so forth.)

In [11]:
feat1 = all_coefs[:,1]
print ( feat1.min() )
print ( feat1.max() )

-20634740.640777323
37018163.782071024


## Ridge regression comes to rescue

Generally, whenever we see weights change so much in response to change in data, we believe the variance of our estimate to be large. Ridge regression aims to address this issue by penalizing "large" weights. (The weights looked quite small, but they are not that small because 'sqft_living' input is in the order of thousands.)

Fit a 15th-order polynomial model on set_1, set_2, set_3, and set_4, this time with a large L2 penalty. Make sure to add "alpha=l2_large_penalty" and "normalize=True" to the parameter list, where the value of l2_large_penalty is given by

In [12]:
l2_large_penalty=1.23e2

In [13]:
# array to hold coefficients from all 4 models
all_coefs = []

# fitting on each subset
for df in (set_1, set_2, set_3, set_4):
    # getting the features list
    poly15_data = polynomial_df(df['sqft_living'], 15)
    y = df['price']

    # normalization: substract mean + divide by l2-norm (standard deviation)
    # note: normalize=True is deprecated in newer versions of sklern
    model = make_pipeline(
        StandardScaler(), # defaults to `with_mean=True, with_std=True`
        Ridge(alpha=l2_large_penalty) # defaults to fit_intercept=True
        )

    # fitting the model
    model.fit(poly15_data, y)

    # printing out the coefficients
    new_coefs = model.named_steps['ridge'].coef_
    all_coefs.append(new_coefs)
    print (new_coefs)

# stack all coefs
all_coefs = np.vstack(all_coefs)

[108478.0990061  145670.20134569  74673.16525436  20086.62767898
   8809.23705503  11260.8023192   12581.2484759    9787.97244607
   3693.45406719  -4315.32236723 -13138.59678306 -22061.32600883
 -30665.99908057 -38732.21028159 -46161.10644053]
[ 69893.7178379  107249.20262787  90268.41498339  34649.23876175
 -15085.42727954 -38161.58423774 -39664.92895888 -30227.44267175
 -17244.22252932  -4440.98599352   6725.97765228  15927.72952913
  23308.30936338  29163.01608684  33798.30811737]
[ 82607.17933907 122785.50722393  87241.05493182  23697.20326315
 -15683.16576216 -28115.05642825 -26233.31448511 -18899.64217342
 -10267.63386235  -1903.85085844   5756.71457328  12700.75494215
  19037.25852497  24886.34956276  30345.98433192]
[ 69493.95814068  89890.1426163   76446.13777659  39238.35167727
   3542.34757567 -16801.24029866 -22335.5084798  -18712.28662812
 -11184.20298755  -3244.04870215   3189.06272237   7280.36207301
   8853.43624953   8090.44707813   5340.64068588]


**QUIZ QUESTION**: For the models learned with regularization in each of these training sets, what are the smallest and largest values you learned for the coefficient of feature power_1?(For the purpose of answering this question, negative numbers are considered "smaller" than positive numbers. So -5 is smaller than -3, and -3 is smaller than 5 and so forth.)

In [14]:
feat1 = all_coefs[:,1]
print ( feat1.min() )
print ( feat1.max() )

89890.1426163009
145670.2013456944


## Selecting an L2 penalty via cross-validation

Just like the polynomial degree, the L2 penalty is a "magic" parameter we need to select. We could use the validation set approach as we did in the last module, but that approach has a major disadvantage: it leaves fewer observations available for training. Cross-validation seeks to overcome this issue by using all of the training set in a smart way.

We will implement a kind of cross-validation called k-fold cross-validation. The method gets its name because it involves dividing the training set into k segments of roughtly equal size. Similar to the validation set method, we measure the validation error with one of the segments designated as the validation set. The major difference is that we repeat the process k times as follows:
- Set aside segment 0 as the validation set, and fit a model on rest of data, and evalutate it on this validation set
- Set aside segment 1 as the validation set, and fit a model on rest of data, and evalutate it on this validation set
- ...
- Set aside segment k-1 as the validation set, and fit a model on rest of data, and evalutate it on this validation set

After this process, we compute the average of the k validation errors, and use it as an estimate of the generalization error. Notice that all observations are used for both training and validation, as we iterate over segments of data.

To estimate the generalization error well, it is crucial to shuffle the training data before dividing them into segments. We reserve 10% of the data as the test set and randomly shuffle the remainder. Le'ts call the shuffled data 'train_valid_shuffled'.

Divide the combined training and validation set into equal segments. Each segment should receive n/k elements, where n is the number of observations in the training set and k is the number of segments. Since the segment 0 starts at index 0 and contains n/k elements, it ends at index (n/k)-1. The segment 1 starts where the segment 0 left off, at index (n/k). With n/k elements, the segment 1 ends at index (n*2/k)-1. Continuing in this fashion, we deduce that the segment i starts at index (n*i/k) and ends at (n*(i+1)/k)-1.

With this pattern in mind, we write a short loop that prints the starting and ending indices of each segment, just to make sure you are getting the splits right.

In [15]:
n = len(train_valid_shuffled)
k = 10 # 10-fold cross-validation

for i in range(k):
    start = (n*i) // k
    end = (n*(i+1)) // k-1
    print ( i, (start, end) )

0 (0, 1938)
1 (1939, 3878)
2 (3879, 5817)
3 (5818, 7757)
4 (7758, 9697)
5 (9698, 11636)
6 (11637, 13576)
7 (13577, 15515)
8 (15516, 17455)
9 (17456, 19395)


Now we are ready to implement k-fold cross-validation. Write a function that computes k validation errors by designating each of the k segments as the validation set. It accepts as parameters (i) k, (ii) l2_penalty, (iii) dataframe containing input features (e.g. poly15_data) and (iv) column of output values (e.g. price). The function returns the average validation error using k segments as validation sets. We shall assume that the input dataframe does not contain the output column.

For each i in [0, 1, ... k-1]:
- Compute starting and ending indices of segment i and call 'start' and 'end'
- Form validation set by taking a slice (start:end+1) from the data.
- Form training set by appending slice (end+1:n) to the end of slice (0:start).
- Train a linear model using training set just formed, with a given l2_penalty
- Compute validation error (RSS) using validation set just formed

In [36]:
def k_fold_cross_validation(k, l2_penalty, data, output):
    # initialization
    n = len(data)
    total_valid_error = 0.0

    # main loop
    for i in range(k):
        
        # calculate start and end index of each set
        start = (n*i) // k
        end = (n*(i+1)) // k-1
        
        # forming validation set
        valid_x = data[start:end+1]
        valid_y = output[start:end+1]

        # forming training set
        train_x = pd.concat( [ data[0:start], data[end+1:n] ] )
        train_y = pd.concat( [ output[0:start], output[end+1:n] ] )

        # normalizing and using penalty
        model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=l2_penalty)
        )

        # training the model
        model.fit(train_x, train_y)

        # computing validation error (RSS)
        predictions = model.predict(valid_x)
        valid_error = sum((valid_y - predictions)**2)
        total_valid_error += valid_error
        
    # computing average validation error
    average_valid_error = total_valid_error / k

    # return statement
    return average_valid_error

Once we have a function to compute the average validation error for a model, we can write a loop to find the model that minimizes the average validation error. Write a loop that does the following:
- We will again be aiming to fit a 15th-order polynomial model using the sqft_living input
- For each l2_penalty in [10^3, 10^3.5, 10^4, 10^4.5, ..., 10^9] (to get this in Python, you can use this Numpy function: np.logspace(3, 9, num=13).): Run 10-fold cross-validation with l2_penalty.
- Report which L2 penalty produced the lowest average validation error.

Note: since the degree of the polynomial is now fixed to 15, to make things faster, you should generate polynomial features in advance and re-use them throughout the loop. Make sure to use train_valid_shuffled when generating polynomial features!

In [37]:
k = 10
data = polynomial_df(train_valid_shuffled['sqft_living'], 15)
output = train_valid_shuffled['price']
average_valid_error = []

for l2_penalty in np.logspace(3, 9, num=13):
    average_valid_error.append( k_fold_cross_validation(k, l2_penalty, data, output) )

In [38]:
average_valid_error

[1.217456428749214e+27,
 9.459869236619387e+26,
 3.5540184931928386e+26,
 5.630686744638756e+24,
 2.497229906272619e+26,
 1.9494959928049324e+26,
 4.880882303352923e+25,
 6.93742333553186e+24,
 7.822689527623127e+23,
 8.133795789961046e+22,
 8.235498377232161e+21,
 8.267649667010294e+20,
 8.27671664364706e+19]

**Quiz Question**: What is the best value for the L2 penalty according to 10-fold validation?

In [40]:
print ( "Answer: 10^9" )

Answer: 10^9


Once you found the best value for the L2 penalty using cross-validation, it is important to retrain a final model on all of the training data using this value of l2_penalty. This way, your final model will be trained on the entire dataset.

In [44]:
train_x = polynomial_df(train_data['sqft_living'], 15)
train_y = train_data['price']
test_x = polynomial_df(test_data['sqft_living'], 15)
test_y = test_data['price']


model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=10^9)
        )

model.fit(train_x, train_y)

predictions = model.predict(test_x)
final_rss = sum((test_y - predictions)**2)

**Quiz Question**: Using the best L2 penalty found above, train a model using all training data. What is the RSS on the TEST data of the model you learn with this L2 penalty?

In [45]:
print ("Answer:", final_rss)

Answer: 134235636097313.14
